In [ ]:
# Common imports for ESBE setup-style notebooks (1/2/3/9).
# Heavy lifting lives in urbanopt_des and lib.helpers; this cell stays terse.
from sys import platform
import copy
import json
import os
from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

# Autoreload dependencies while iterating on wrapper / helper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

# Weather data is copied over from the activity_03/coincident project. No
# need to update the weather information.

### Activity 05: REopt

In [ ]:
# Copy activity_03/coincident to activity_05/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_03") / "coincident"
activity_3_dir = paths.activity_dir("activity_05")
dest_dir = activity_3_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_05/coincident
uo_coincident = UO(activity_3_dir, "coincident", template_dir=template_data_dir)

In [ ]:
# Create the scenario mapper file with the REopt assumptions enabled.
# Confirm REopt assumptions show up in REopt_baseline_scenario after this runs.
uo_coincident.create_reopt_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)

In [ ]:
# Run the REopt baseline scenario.
uo_coincident.run("class_project_coincident.json", "REopt_baseline_scenario.csv")

In [ ]:
# Build optimized REopt assumptions JSON + scenario CSV that point at it.
activity_3_analysis_dir = paths.activity_dir("activity_05")
reopt_dir = activity_3_analysis_dir / "coincident" / "reopt"
base_assumptions_path = reopt_dir / "multiPV_assumptions.json"

# 1. Load base assumptions and apply changes
with open(base_assumptions_path) as f:
    base = json.load(f)
optimized = copy.deepcopy(base)

# --- Financial -----------------------------------------------------------
optimized["Financial"]["elec_cost_escalation_rate_fraction"] = 0.05  # 0.026 → 0.05
optimized["Financial"]["offtaker_discount_rate_fraction"] = 0.04  # 0.081 → 0.04
optimized["Financial"]["emissions_cost_per_mt_co2"] = 100.0  # add carbon tax

# --- PV ------------------------------------------------------------------
for pv_array in optimized["PV"]:
    pv_array["installed_cost_per_kw"] = 1000.0  # $2,000 → $1,000 per kW

# --- ElectricTariff: $25/kW monthly demand charge ------------------------
optimized["ElectricTariff"]["add_monthly_rates_to_urdb_rate"] = True
optimized["ElectricTariff"]["monthly_demand_rates"] = [25.0] * 12

# --- ElectricUtility: allow export so PV surplus can charge battery ------
optimized["ElectricUtility"]["net_metering_limit_kw"] = 1_000_000.0
optimized["ElectricUtility"]["interconnection_limit_kw"] = 1_000_000.0

# --- ElectricStorage: ITC + ATB 2024 cost defaults -----------------------
optimized["ElectricStorage"].update(
    {
        "min_kw": 0.0,
        "max_kw": 1_000_000.0,
        "min_kwh": 0.0,
        "max_kwh": 1_000_000.0,
        "installed_cost_per_kw": 775.0,
        "installed_cost_per_kwh": 388.0,
        "replace_cost_per_kw": 410.0,
        "replace_cost_per_kwh": 220.0,
        "battery_replacement_year": 10,
        "inverter_replacement_year": 10,
        "total_itc_fraction": 0.3,
        "macrs_option_years": 7,
        "macrs_bonus_fraction": 0.6,
        "macrs_itc_reduction": 0.5,
        "soc_min_fraction": 0.2,
        "soc_init_fraction": 0.5,
        "internal_efficiency_fraction": 0.975,
        "inverter_efficiency_fraction": 0.96,
        "rectifier_efficiency_fraction": 0.96,
        "can_grid_charge": True,
    }
)

# --- Wind: distributed-wind, ATB 2024 commercial defaults ----------------
optimized["Wind"] = {
    "min_kw": 0.0,
    "max_kw": 1_000_000.0,
    "size_class": "commercial",
    "installed_cost_per_kw": 3013.0,
    "om_cost_per_kw": 40.0,
    "macrs_option_years": 5,
    "macrs_bonus_fraction": 0.6,
    "federal_itc_fraction": 0.3,
    "can_net_meter": True,
    "can_wholesale": False,
    "can_export_beyond_nem_limit": False,
}

# 2. Write the new assumptions file
new_assumptions_name = "multiPV_optimized_assumptions.json"
new_assumptions_path = reopt_dir / new_assumptions_name
with open(new_assumptions_path, "w") as f:
    json.dump(optimized, f, indent=2)
print(f"Wrote: {new_assumptions_path}")

# 3. Build new scenario CSV pointing to the new assumptions file
base_mapper = activity_3_analysis_dir / "coincident" / "REopt_baseline_scenario.csv"
new_mapper = activity_3_analysis_dir / "coincident" / "REopt_optimized_scenario.csv"
new_mapper.write_text(
    base_mapper.read_text().replace("multiPV_assumptions.json", new_assumptions_name)
)
print(f"Wrote: {new_mapper}")

# 4. Summary of parameter changes
print("\nParameter changes applied:")
print(
    f"  Financial.elec_cost_escalation_rate   : {base['Financial']['elec_cost_escalation_rate_fraction']} → {optimized['Financial']['elec_cost_escalation_rate_fraction']}"
)
print(
    f"  Financial.offtaker_discount_rate      : {base['Financial']['offtaker_discount_rate_fraction']} → {optimized['Financial']['offtaker_discount_rate_fraction']}"
)
print(
    f"  Financial.emissions_cost_per_mt_co2   : {optimized['Financial']['emissions_cost_per_mt_co2']}"
)
print(
    f"  PV[*].installed_cost_per_kw           : {base['PV'][0]['installed_cost_per_kw']} → {optimized['PV'][0]['installed_cost_per_kw']}"
)
print(
    f"  ElectricTariff.monthly_demand_rates   : ${optimized['ElectricTariff']['monthly_demand_rates'][0]}/kW × 12"
)
print(
    f"  ElectricUtility.net_metering_limit_kw : {base['ElectricUtility']['net_metering_limit_kw']} → {optimized['ElectricUtility']['net_metering_limit_kw']:,.0f}"
)
print(
    f"  ElectricStorage.total_itc_fraction    : {base['ElectricStorage']['total_itc_fraction']} → {optimized['ElectricStorage']['total_itc_fraction']}"
)
print(
    f"  ElectricStorage.installed_cost_per_kw : {base['ElectricStorage']['installed_cost_per_kw']} → {optimized['ElectricStorage']['installed_cost_per_kw']}"
)
print(
    f"  ElectricStorage.installed_cost_per_kwh: {base['ElectricStorage']['installed_cost_per_kwh']} → {optimized['ElectricStorage']['installed_cost_per_kwh']}"
)
print(
    f"  Wind.max_kw                           : {base['Wind']['max_kw']} → {optimized['Wind']['max_kw']:,.0f}"
)
print(
    f"  Wind.installed_cost_per_kw            : {base['Wind']['installed_cost_per_kw']} → {optimized['Wind']['installed_cost_per_kw']}"
)
print(f"  Wind.size_class                       : {optimized['Wind']['size_class']}")

In [ ]:
# Copy the already-run REopt baseline outputs over to the optimized scenario
# directory so we don't burn cycles re-running shared simulation work.
source_dir = activity_3_analysis_dir / "coincident" / "run" / "reopt_baseline_scenario"
dest_dir = activity_3_analysis_dir / "coincident" / "run" / "reopt_optimized_scenario"
copy_activity_to_new_location(source_dir, dest_dir, overwrite=True)

In [ ]:
# Post-process the REopt scenarios (scenario-level summary).
# SSL_CERT_FILE: force embedded Ruby/URBANopt to use a valid CA bundle on NLR macOS. This works on non-NLR systems too.

# only check this if on macos system.
if platform == "darwin":
    os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
    print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

for scenario_csv in ("REopt_baseline_scenario.csv", "REopt_optimized_scenario.csv"):
    uo_coincident.process_reopt_scenario(
        "class_project_coincident.json",
        scenario_csv,
        individual_features=False,
    )

In [ ]:
# Post-process the REopt scenarios (scenario-level summary).
# SSL_CERT_FILE: force embedded Ruby/URBANopt to use a valid CA bundle on NLR macOS. This works on non-NLR systems too.

# only check this if on macos system.
if platform == "darwin":
    os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
    print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

# Optional: per-feature REopt breakdown — slow (~3 min × 23 buildings ≈ 70 min; depending on REopt demand).
os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=True,
)